In [8]:
import os
import pandas as pd

# Load Ratios.xlsx (adapt path if needed)
PATH = "./Ford_Ratios_Local.xlsx"
spreadsheet = os.path.abspath(PATH)
df = pd.read_excel(spreadsheet, sheet_name=None)

In [9]:
# Extract 2024 data for Ford (from '2024' sheet)
data = df['2024'].set_index('Variable')['Value'].to_dict()  # Assume column A = Variable, B = Value

In [10]:
# Extract 2024 data for Ford (from '2024' sheet)
df2020 = pd.read_excel(spreadsheet, sheet_name="2020").set_index("Variable")["Value"]
df2021 = pd.read_excel(spreadsheet, sheet_name="2021").set_index("Variable")["Value"]
df2022 = pd.read_excel(spreadsheet, sheet_name="2022").set_index("Variable")["Value"]
df2023 = pd.read_excel(spreadsheet, sheet_name="2023").set_index("Variable")["Value"]
df2024 = pd.read_excel(spreadsheet, sheet_name="2024").set_index("Variable")["Value"]

In [ ]:
    shares_outstanding_m = 4040           # million shares (Ford 10-K 2024.0bn actual + 2.9bn dilutive)
    stock_price_dec2025 = 13.78          # closing price you want to use
    market_cap = shares_outstanding_m * stock_price_dec2025   # $93.15 billion
    risk_free_rate = 0.0420               # 10-yr US Treasury yield
    equity_risk_premium = 0.0433          # Damodaran Jan 2025
    beta_5yr = 1.61                       # Bloomberg / Yahoo Finance 5-yr monthly beta vs S&P500


    net_debt_2024 = df2024["TotalDebt2024"] - df2024["CashAndMktSec2024"]
    total_capital = market_cap + net_debt_2024
    equity_weight = market_cap / total_capital
    debt_weight   = net_debt_2024 / total_capital

    interest_expense_2024   = df2024["InterestExpense2024"]
    avg_debt_2024           = (df2023["TotalDebt2023"] + df2024["TotalDebt2024"]) / 2
    pre_tax_cost_of_debt    = interest_expense_2024 / avg_debt_2024   # ≈ 5.8–6.0 %
    tax_rate_2024           = df2024["TaxExpense2024"] / df2024["PreTaxIncome2024"]
    print(f"Avg Debt: {avg_debt_2024}")
    print(f"Effective Tax Rate: {tax_rate_2024}")

    # Cost of Equity (CAPM)
    re = risk_free_rate + beta_5yr * (equity_risk_premium - risk_free_rate)
    print(f"Required Return: {re}")

    calculated_equity_risk_premium = re - risk_free_rate 
    print(f"Calculated Risk Premium: {re}")


    # WACC
    wacc = (equity_weight * re) + (debt_weight * pre_tax_cost_of_debt * (1 - tax_rate_2024))
    print(f"Calculated Cost of Equity: {re:.3%}")
    print(f"Calculated WACC:           {wacc:.3%}")

    # Growth rates – derived from your data
    revenue_cagr_2020_2024 = ((df2024["Revenue2024"] / df2020["Revenue2020"]) ** (1/5)) - 1
    g_short = max(0.025, min(0.06, revenue_cagr_2020_2024 * 0.6))   # conservative haircut
    g_terminal = 0.023   # long-term GDP + inflation (Fed dot plot Dec 2025)

    print(f"Revenue CAGR 2020-2024: {revenue_cagr_2020_2024:.1%} → using short-term growth {g_short:.1%}")

    # Key inputs from workbook
    net_income = df2024['NetIncomeAttrib2024']
    operating_income = df2024['OpsIncome2024']
    nopat = df2024['NOPAT2024']
    depreciation = df2024['DepreciationAmort2024']
    capex = abs(df2024['CAPEX2024'])  # From cash flow
    delta_owc = abs(df2023["OWC2023"] - df2024["OWC2024"]) # From workbook
    net_debt_issuance = df2024['NetDebtIssuance2024']  # From cash flow
    book_equity_begin = df2024['EquityAttrib2024']  # 2023 equity
    dividends = abs(df2024["CashDividendsPaid2024"])  # From workbook
    revenue = df2024['Revenue2024']
    payout_ratio = dividends / net_income
    # Assumptions (from essay)
    years = 5

    # Projections (simple linear from workbook trends)
    proj_net_income = [net_income * (1 + g_short) ** t for t in range(1, years+1)]
    proj_div = [dividends * (1 + g_short) ** t for t in range(1, years+1)]
    proj_fcfe = [proj_net_income[t-1] - capex * (1 + g_short) ** t - delta_owc * (1 + g_short) ** t + net_debt_issuance * (1 + g_short) ** t for t in range(1, years+1)]
    proj_fcff = [nopat * (1 + g_short) ** t + depreciation * (1 + g_short) ** t - capex * (1 + g_short) ** t - delta_owc * (1 + g_short) ** t for t in range(1, years+1)]
    proj_ri = [proj_net_income[t-1] - re * book_equity_begin * (1 + g_short) ** (t-1) for t in range(1, years+1)]

    # DDM (Gordon)
    ddm_d1 = proj_div[0]
    ddm_value = ddm_d1 / (re - g_terminal)
    ddm_per_share = ddm_value / shares_outstanding_m  # 6.9bn shares

    proj_fcff = []
    for t in range(1, years + 1):
        fcff_t = (
            net_income * (1 + g_short)**t
            + depreciation * (1 + g_short)**t
            + interest_expense_2024 * (1 - tax_rate_2024) * (1 + g_short)**t   # after-tax interest add-back
            - capex * (1 + g_short)**t
            - delta_owc * (1 + g_short)**t
        )
        proj_fcff.append(fcff_t)

    # PV of explicit period
    pv_fcff_explicit = sum(proj_fcff[i] / (1 + wacc)**(i+1) for i in range(5))

    # Terminal value at end of 2029
    terminal_fcff = proj_fcff[-1] * (1 + g_terminal) / (wacc - g_terminal)
    pv_terminal_fcff = terminal_fcff / (1 + wacc)**5

    # Enterprise Value
    enterprise_value = pv_fcff_explicit + pv_terminal_fcff

    net_debt_2024 = df2024["TotalDebt2024"] - df2024["CashAndMktSec2024"]
    equity_value_fcff = enterprise_value - net_debt_2024
    fcff_per_share = equity_value_fcff / shares_outstanding_m

    # FCFE
    pv_fcfe = sum(proj_fcfe[t-1] / (1 + re)**t for t in range(1, years+1))
    terminal_fcfe = proj_fcfe[-1] * (1 + g_terminal) / (re - g_terminal)
    pv_terminal_fcfe = terminal_fcfe / (1 + re)**years

    equity_value_fcfe = pv_fcfe + pv_terminal_fcfe
    fcfe_per_share = equity_value_fcfe / shares_outstanding_m

    # Residual Income
    pv_ri = sum(proj_ri[t-1] / (1 + re) ** t for t in range(1, years+1))
    terminal_ri = proj_ri[-1] * (1 + g_terminal) / (re - g_terminal)
    pv_terminal_ri = terminal_ri / (1 + re) ** years
    ri_value = book_equity_begin + pv_ri + pv_terminal_ri
    ri_per_share = ri_value / shares_outstanding_m

    # Output
    print(f"Ford 2024 DDM Per Share: ${ddm_per_share:.2f}")
    print(f"FCFF Equity Value: ${abs(equity_value_fcff)/1000:.1f} bn → ${abs(fcff_per_share):.2f}/share")
    print(f"FCFE Equity Value: ${abs(equity_value_fcfe)/1000:.1f} bn → ${abs(fcfe_per_share):.2f}/share")
    print(f"Ford 2024 RI Per Share: ${ri_per_share:.2f}")

Avg Debt: 153876.5
Effective Tax Rate: 0.1851237384211254
Required Return: 0.04409299999999999
Calculated Risk Premium: 0.04409299999999999
Calculated Cost of Equity: 4.409%
Calculated WACC:           5.799%
Revenue CAGR 2021-2024: 7.8% → using short-term growth 4.7%
Ford 2024 DDM Per Share: $38.30
FCFF Equity Value: $131.0 bn → $32.42/share
FCFE Equity Value: $334.8 bn → $82.86/share
Ford 2024 RI Per Share: $64.59


In [20]:
import openpyxl
# Create the DataFrame again (clean)
results = [
    {"Model": "Dividend Discount Model", 
     "Formula": "D₁ / (rₑ − g)", 
     "Value_per_Share": ddm_per_share},
    
    {"Model": "Free Cash Flow to Equity", 
     "Formula": "Σ FCFEₜ/(1+rₑ)ᵗ + TVₑ", 
     "Value_per_Share": abs(fcfe_per_share)},
    
    {"Model": "Free Cash Flow to Firm", 
     "Formula": "Σ FCFFₜ/(1+WACC)ᵗ + TV − Net Debt", 
     "Value_per_Share": abs(fcff_per_share)},
    
    {"Model": "Residual Income", 
     "Formula": "Book Value + Σ RIₜ/(1+rₑ)ᵗ + TV_RI", 
     "Value_per_Share": ri_per_share},
]

df_valuation = pd.DataFrame(results)

# Add vs Market column
market_price = 13.78
df_valuation["vs_Market_%"] = ((df_valuation["Value_per_Share"] - market_price) / market_price * 100).round(1)
df_valuation["vs_Market_%"] = df_valuation["vs_Market_%"].astype(str) + "%"

# Combine Model + Formula in one column
df_valuation["Model (Formula)"] = df_valuation["Model"] + " (" + df_valuation["Formula"] + ")"
df_valuation = df_valuation[["Model (Formula)", "Value_per_Share", "vs_Market_%"]]

# Rename columns correctly (now 3 columns → 3 names)
df_valuation.columns = ["Model (Formula)", "Intrinsic Value per Share ($)", "vs. Market $13.50"]

# Save to Excel
output_file = "Ford_Valuation.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_valuation.to_excel(writer, sheet_name='Absolute', index=False, startrow=1)

    # Make header bold
    worksheet = writer.sheets['Absolute']
    for cell in worksheet[2]:
        cell.font = openpyxl.styles.Font(bold=True)

print(f"\nSUCCESS! File saved: {output_file}")
print("Sheet: Absolute")
print(df_valuation.to_string(index=False))


SUCCESS! File saved: Ford_Valuation.xlsx
Sheet: Absolute
                                           Model (Formula)  Intrinsic Value per Share ($) vs. Market $13.50
                   Dividend Discount Model (D₁ / (rₑ − g))                      38.299302            177.9%
          Free Cash Flow to Equity (Σ FCFEₜ/(1+rₑ)ᵗ + TVₑ)                      82.864737            501.3%
Free Cash Flow to Firm (Σ FCFFₜ/(1+WACC)ᵗ + TV − Net Debt)                      32.424466            135.3%
      Residual Income (Book Value + Σ RIₜ/(1+rₑ)ᵗ + TV_RI)                      64.585153            368.7%
